In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm
import plotly
from plotly.graph_objs import Surface
import yfinance as yf
import math
from bs4 import BeautifulSoup # library to parse HTML documents

import requests
import json

In [2]:
tickers = pd.read_csv('extracted_tickers.csv')['Ticker']
len(tickers)

3029

In [3]:
# get the response in the form of html
wikiurl="https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
#table_class="wikitable sortable components"
response=requests.get(wikiurl)
print(response.status_code)

# parse data from the html into a beautifulsoup object
soup = BeautifulSoup(response.text, 'html.parser')
indiatable=soup.find_all('table',{'class':"wikitable"})

df_sp500 = pd.read_html(str(indiatable[0]))
# convert list to dataframe
df_sp500=pd.DataFrame(df_sp500[0])
# print(df_sp500.head())

companies = df_sp500["Symbol"].values
companies

200


array(['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL',
       'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT',
       'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AAL', 'AEP',
       'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI',
       'ANSS', 'AON', 'APA', 'AAPL', 'AMAT', 'APTV', 'ACGL', 'ADM',
       'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVB',
       'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BK', 'BBWI', 'BAX', 'BDX',
       'BRK.B', 'BBY', 'BIO', 'TECH', 'BIIB', 'BLK', 'BX', 'BA', 'BKNG',
       'BWA', 'BXP', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF.B', 'BLDR',
       'BG', 'CDNS', 'CZR', 'CPT', 'CPB', 'COF', 'CAH', 'KMX', 'CCL',
       'CARR', 'CTLT', 'CAT', 'CBOE', 'CBRE', 'CDW', 'CE', 'COR', 'CNC',
       'CNP', 'CF', 'CHRW', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB',
       'CHD', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME',
       'CMS', 'KO', 'CTSH', 'CL', 'CMCSA', 'CMA', 'CAG', 'COP', 'ED',
  

In [4]:
df_sp500

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
499,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
500,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927
501,ZION,Zions Bancorporation,Financials,Regional Banks,"Salt Lake City, Utah",2001-06-22,109380,1873


In [5]:
tickers

0         MSFT
1         AMZN
2         BRKB
3          JPM
4           FB
         ...  
3024        NH
3025        --
3026     RTYZ8
3027      ESZ8
3028    P5N994
Name: Ticker, Length: 3029, dtype: object

In [6]:
tickers_without_sp500 = list(set(tickers) - set(companies))
len(tickers_without_sp500)

2566

## Process S&P500 tickers

In [7]:
for comp in companies:
    with open('tickers.csv', 'a', encoding='utf-8') as f:
        sect = df_sp500[df_sp500['Symbol'] == comp]['GICS Sector']
        sect_value = sect.values[0]
        f.write(f'{comp},{sect_value}\n')

In [8]:
tickers_done = list(pd.read_csv('tickers.csv')['Ticker'])
len(tickers_done)

1584

In [10]:
tickers_without_sp500 = list(set(tickers) - set(tickers_done))
len(tickers_without_sp500)

2493

In [11]:
api_key = '2d1d4673e63477d196d2eeeea2db8bcba914256992b5c17f1ebcaab9d508f6b9'
tickers_done = companies

for name in tickers_without_sp500:
    try:
        res = requests.get(f'https://api.sec-api.io/mapping/ticker/{name}?token={api_key}').json()[0]
        sect_value = res['sector']

        with open('tickers.csv', 'a', encoding='utf-8') as f:
            f.write(f'{name},{sect_value}\n')
        tickers_done.append(name)
    except:
        continue
